In [5]:
%reload_ext dotenv
%dotenv ../../05_src/.secrets

In [ ]:
import requests
import os
from openai import OpenAI


# Load keys
ONET_API_KEY = os.getenv("ONET_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

BASE_URL = "https://api-v2.onetcenter.org/online"
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('OPENAI_API_KEY')})


# O*NET SEARCH FUNCTION
def search_occupation(keyword: str, max_results: int = 3):
    url = f"{BASE_URL}/search"
    
    headers = {
        "X-API-Key": ONET_API_KEY,
        "Accept": "application/json"
    }
    
    params = {
        "keyword": keyword,
        "end": max_results
    }
    
    response = requests.get(url, params=params, headers=headers)
    response.raise_for_status()
    data = response.json()
    
    if "occupation" in data:
        return [(occ["code"], occ["title"]) for occ in data["occupation"]]
    
    return []


# GET OCCUPATION DETAILS

def get_occupation(code: str):
    url = f"{BASE_URL}/occupations/{code}"
    
    headers = {
        "X-API-Key": ONET_API_KEY,
        "Accept": "application/json"
    }
    
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    
    return response.json()


# LLM: EXTRACT JOB TITLE

def extract_job_title(user_sentence: str) -> str:
    """
    Uses GPT to extract a clean job title from a user-provided sentence.
    """
    prompt = f"""
    You are an occupational analyst.
    Extract the main job title from the following sentence.
    Respond with **only the job title**, no extra words or punctuation.
    
    Sentence: "{user_sentence}"
    """
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You extract job titles from text."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    
    job_title = response.choices[0].message.content.strip()
    return job_title


# LLM: DESCRIBE WORK ACTIVITIES

def describe_work_activities(title, description):
    prompt = f"""
    You are an occupational analyst.
    
    Based on the following O*NET occupation description,
    clearly explain the core work activities performed in this job.
    
    Occupation: {title}
    
    Description:
    {description}
    
    Provide:
    - 5–8 bullet points
    - Clear, concrete work activities
    - Professional tone
    """
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You analyze occupations professionally."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    
    return response.choices[0].message.content


# CONVERSATIONAL INTERFACE
def chat():
    print("👋 Hello! I can describe the work activities of any occupation.")
    
    while True:
        user_input = input("\nDescribe the job you want to know about (type 'quit' to exit): ")
        
        if user_input.lower() == "quit":
            print("Goodbye! 👋")
            break
        
        # 1️⃣ Extract job title from user sentence
        job_title = extract_job_title(user_input)
        print(f"🔹 Extracted job title: {job_title}")
        
        # 2️⃣ Search O*NET with extracted title
        results = search_occupation(job_title)
        
        if not results:
            print("Sorry, I couldn't find that occupation in O*NET.")
            continue
        
        print("\nI found the following occupations:")
        for i, (code, title) in enumerate(results):
            print(f"{i+1}. {title} ({code})")
        
        choice = input("\nSelect a number: ")
        
        try:
            selected_code, selected_title = results[int(choice) - 1]
        except:
            print("Invalid selection.")
            continue
        
        print("\nRetrieving occupation details...\n")
        
        data = get_occupation(selected_code)
        description = data["description"]
        
        analysis = describe_work_activities(selected_title, description)
        
        print("📋 Work Activities:\n")
        print(analysis)



# RUN THE CHATBOT
chat()


👋 Hello! I can describe the work activities of any occupation.
🔹 Extracted job title: swimming instructor

I found the following occupations:
1. Self-Enrichment Teachers (25-3021.00)
2. Coaches and Scouts (27-2022.00)
3. Recreation and Fitness Studies Teachers, Postsecondary (25-1193.00)

Retrieving occupation details...

📋 Work Activities:

**Core Work Activities of Recreation and Fitness Studies Teachers, Postsecondary:**

- **Course Development and Instruction:** Design and deliver engaging curricula for courses related to recreation, leisure, and fitness studies, ensuring alignment with educational standards and student learning objectives.

- **Assessment and Evaluation:** Develop and implement assessment tools to evaluate student performance and understanding, providing constructive feedback to foster academic growth.

- **Research and Scholarship:** Conduct research in areas related to recreation and fitness studies, contributing to the body of knowledge in the field and publish

In [ ]:
#v2 with GRADIO

import gradio as gr
# Load keys
ONET_API_KEY = os.getenv("ONET_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

BASE_URL = "https://api-v2.onetcenter.org/online"
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('OPENAI_API_KEY')})


# ===============================
# 3️⃣ O*NET SEARCH
# ===============================
def search_occupation(keyword: str, max_results: int = 3):
    url = f"{BASE_URL}/search"
    headers = {"X-API-Key": ONET_API_KEY, "Accept": "application/json"}
    params = {"keyword": keyword, "end": max_results}
    response = requests.get(url, params=params, headers=headers)
    response.raise_for_status()
    data = response.json()
    if "occupation" in data:
        return [(occ["code"], occ["title"]) for occ in data["occupation"]]
    return []


# ===============================
# 4️⃣ GET OCCUPATION DETAILS
# ===============================
def get_occupation(code: str):
    url = f"{BASE_URL}/occupations/{code}"
    headers = {"X-API-Key": ONET_API_KEY, "Accept": "application/json"}
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    return response.json()


# ===============================
# 5️⃣ LLM: EXTRACT JOB TITLE
# ===============================
def extract_job_title(user_sentence: str) -> str:
    prompt = f"""
    You are an occupational analyst.
    Extract the main job title from the following sentence.
    Respond with **only the job title**, no extra words or punctuation.
    
    Sentence: "{user_sentence}"
    """
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You extract job titles from text."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    job_title = response.choices[0].message.content.strip()
    return job_title


# ===============================
# 6️⃣ LLM: DESCRIBE WORK ACTIVITIES
# ===============================
def describe_work_activities(title, description):
    prompt = f"""
    You are an occupational analyst.
    
    Based on the following O*NET occupation description,
    clearly explain the core work activities performed in this job.
    
    Occupation: {title}
    
    Description:
    {description}
    
    Provide:
    - 5–8 bullet points
    - Clear, concrete work activities
    - Professional tone
    """
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You analyze occupations professionally."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    return response.choices[0].message.content


# ===============================
# 7️⃣ GRADIO FUNCTION
# ===============================
def analyze_job(user_sentence: str) -> str:
    # 1️⃣ Extract job title
    job_title = extract_job_title(user_sentence)
    
    # 2️⃣ Search O*NET
    results = search_occupation(job_title)
    if not results:
        return "❌ No occupations found in O*NET."
    
    # 3️⃣ Use the first result
    selected_code, selected_title = results[0]
    
    # 4️⃣ Get O*NET description
    data = get_occupation(selected_code)
    description = data["description"]
    
    # 5️⃣ Generate work activities
    analysis = describe_work_activities(selected_title, description)
    
    # 6️⃣ Return nicely formatted markdown
    return f"**Job Title:** {selected_title}\n\n**Work Activities:**\n{analysis}"


# ===============================
# 8️⃣ LAUNCH GRADIO INTERFACE
# ===============================
iface = gr.Interface(
    fn=analyze_job,
    inputs=gr.Textbox(lines=2, placeholder="Describe a job in a sentence..."),
    outputs="markdown",
    title="O*NET Job Analyzer",
    description="Type a sentence about any job. The system extracts the job title, searches O*NET, and summarizes the work activities."
)

iface.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
